In [30]:
from langgraph.graph import StateGraph, END
from langchain_openai import AzureChatOpenAI
from typing import TypedDict, List, Any
from dotenv import load_dotenv

load_dotenv()

True

In [31]:
# --- State Definition ---
class State(TypedDict):
    query: str
    candidate: str
    feedback: str
    score: float
    iteration: int

llm = AzureChatOpenAI(model="gpt-4o-mini", temperature=0, api_version='2024-08-01-preview')


In [32]:
# --- Agents ---

def initializer(state: State) -> State:
    # state["candidate"] = llm.invoke(state["query"]).content
    return {"candidate": llm.invoke(state["query"]).content}

def optimizer(state: State) -> State:
    """Generate or refine a candidate solution."""
    prompt = f"""
    Candidate so far: {state.get('candidate', 'None')}
    Feedback: {state.get('feedback', 'None')}
    Please propose a refined solution.
    """
    result = llm.invoke(prompt)
    # state["candidate"] = result.content
    state["iteration"] += 1
    return {"candidate": result.content}

def evaluator(state: State) -> State:
    """Evaluate candidate and give feedback."""
    prompt = f"""
    Candidate: {state['candidate']}
    Evaluate this solution from 1 to 10 on the basis of how well the reponse can be understood by a school child and explain why.
    """
    result = llm.invoke(prompt)
    
    # parse evaluation (naive example)
    feedback = result.content
    score = 0
    for token in feedback.split():
        if token.isdigit():
            score = float(token)
            break
    
    # state["feedback"] = feedback
    # state["score"] = score
    return {"feedback": feedback , "score": score}

# --- Conditional Routing ---
def should_continue(state: State) -> str:
    if state["score"] >= 8:
        return END
    if state["iteration"] >= 5:
        return END
    return "optimizer"



In [33]:
# --- Build LangGraph ---
workflow = StateGraph(State)

workflow.add_node("initializer", initializer)
workflow.add_node("optimizer", optimizer)
workflow.add_node("evaluator", evaluator)

workflow.set_entry_point("initializer")
workflow.add_edge("initializer", "evaluator")
workflow.add_edge("optimizer", "evaluator")
workflow.add_conditional_edges("evaluator", should_continue)

graph = workflow.compile()

In [34]:
# --- Run ---
initial_state: State = {"query":"What is differentiation?", "candidate": "", "feedback": "", "score": 0, "iteration": 0}

# final_state = graph.invoke(initial_state)
# print("Final Candidate:", final_state["candidate"])
# print("Final Score:", final_state["score"])
# print("Feedback:", final_state["feedback"])

for s in graph.stream(initial_state):
    print(s)

{'initializer': {'candidate': "Differentiation is a fundamental concept in calculus that refers to the process of finding the derivative of a function. The derivative measures how a function changes as its input changes, essentially providing the rate of change or the slope of the function at a given point.\n\nIn more formal terms, if you have a function \\( f(x) \\), the derivative \\( f'(x) \\) (or \\( \\frac{df}{dx} \\)) represents the limit of the average rate of change of the function as the interval approaches zero. Mathematically, this is expressed as:\n\n\\[\nf'(x) = \\lim_{h \\to 0} \\frac{f(x + h) - f(x)}{h}\n\\]\n\nDifferentiation has various applications, including:\n\n1. **Finding slopes**: It helps in determining the slope of a curve at a specific point.\n2. **Optimization**: It is used to find maximum and minimum values of functions.\n3. **Motion analysis**: In physics, differentiation is used to analyze motion, such as velocity and acceleration.\n4. **Modeling change**: